# AquaVision - Classifier Train + Test on **Kaggle** (Fish4Knowledge, unaugmented baseline)

Trains the **YOLO11-cls** species classifier with **no augmentation**, tests it (top-1 + **macro-F1**).

**Kaggle setup (right-hand Session options):** Accelerator = **GPU T4 x2**, Internet = **On**. Attach the
**`aquavision-scripts`** dataset (Add Data). For disconnect-proof training, run via **Save Version ->
Save & Run All (Commit)** - it executes headless to completion and persists `/kaggle/working`.

Inputs are read-only (`/kaggle/input/...`); all outputs go under `/kaggle/working/` and appear in the
**Output** tab after a Commit (that's where you download `best.pt` - no Drive, no files.download).

## 0 - GPU + install
Kaggle ships CUDA torch + sklearn + pyyaml; only `ultralytics` is installed. Do NOT pip-install torch (that pulls the CPU wheel).

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics
import torch
assert torch.cuda.is_available(), "No CUDA torch - set Accelerator=GPU and do not upgrade torch"
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU", torch.cuda.get_device_name(0))

## 1 - Scripts + config
Inputs are read-only, so copy the scripts into `/kaggle/working` and `chdir` there.

In [ ]:
import os, shutil, glob
hits = glob.glob("/kaggle/input/aquavision-scripts") + glob.glob("/kaggle/input/*/*/aquavision-scripts")
assert hits, "aquavision-scripts not attached - Add Input -> aquavision-scripts (see !ls /kaggle/input)"
SCRIPTS = hits[0]; print("SCRIPTS =", SCRIPTS, "| contains:", sorted(os.listdir(SCRIPTS)))
NEED = ["f4k_prepare.py", "underwater_aug.py"]
missing = [f for f in NEED if not os.path.exists(f"{SCRIPTS}/{f}")]
assert not missing, (f"{missing} not in the aquavision-scripts dataset - open the dataset -> New Version -> "
                     f"add the file(s), then update the attached input to the new version")
REPO = "/kaggle/working/aquavision"; os.makedirs(REPO, exist_ok=True)
for f in NEED:
    shutil.copy(f"{SCRIPTS}/{f}", f"{REPO}/{f}")
os.chdir(REPO); print("cwd:", os.getcwd(), "| have:", sorted(os.listdir(REPO)))

F4K  = "/kaggle/working/data/fish4knowledge/fish_image"   # 23 species subfolders
WORK = "/kaggle/working/work"
RUNS = "/kaggle/working/runs"
EPOCHS_CLS = 60

## 2 - Fetch Fish4Knowledge (GitHub, Internet must be On)

In [ ]:
import os; os.makedirs("/kaggle/working/data", exist_ok=True)
!git clone --depth 1 https://github.com/Callmewuxin/fish4konwledge /kaggle/working/data/fish4knowledge
print("F4K ->", F4K)

## 3 - Prepare classifier data (no synthetic aug, no capping)
70/20/10 per-class group-aware split + leakage check; flags switch all balancing/aug OFF so `train/`
holds the real crops at their natural (imbalanced) distribution.

In [ ]:
!python f4k_prepare.py --f4k "{F4K}" --out "{WORK}/cls_baseline" --max_aug_ratio 0 --extra_aug 0 --max_per_class 100000 --ceil_mult 1000 --ref_det ""

## 4 - Train classifier (augmentation OFF)
`save_period=5` writes a checkpoint every 5 epochs (useful if you run interactively).

In [ ]:
from ultralytics import YOLO
cls = YOLO("yolo11s-cls.pt")   # ImageNet-pretrained; no data augmentation
# exist_ok=True: always write to runs/species_baseline (no silent species_baseline-2 folders)
cls.train(data=f"{WORK}/cls_baseline", epochs=EPOCHS_CLS, imgsz=224, batch=64,
          device=0, workers=2, project=RUNS, name="species_baseline", exist_ok=True, save_period=5,
          hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, degrees=0.0, translate=0.0, scale=0.0,
          fliplr=0.0, erasing=0.0, auto_augment=None)
CLS_DIR = str(cls.trainer.save_dir); print("run folder:", CLS_DIR)

## 5 - Test: top-1 (Ultralytics) + macro-F1 (sklearn)

In [ ]:
import json
from pathlib import Path
from sklearn.metrics import f1_score, classification_report, confusion_matrix

cm = cls.val(data=f"{WORK}/cls_baseline", split="test", project=RUNS, name="species_baseline_test", exist_ok=True)
print("CLASSIFIER (test):  top-1:", round(cm.top1, 4), "| top-5:", round(cm.top5, 4))

names = cls.names; name2id = {v: k for k, v in names.items()}; labels = sorted(names)
exts = {".jpg", ".jpeg", ".png", ".bmp"}
imgs = [p for p in (Path(WORK) / "cls_baseline" / "test").rglob("*") if p.suffix.lower() in exts]
y_true, y_pred = [], []
for i in range(0, len(imgs), 256):
    chunk = imgs[i:i + 256]
    for p, r in zip(chunk, cls.predict([str(x) for x in chunk], imgsz=224, verbose=False)):
        y_true.append(name2id[p.parent.name]); y_pred.append(int(r.probs.top1))
macro_f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
report = classification_report(y_true, y_pred, labels=labels,
                               target_names=[names[i] for i in labels], zero_division=0)
print(f"macro-F1 (test): {macro_f1:.4f}\n"); print(report)

# save everything next to the weights so one folder holds the whole result
json.dump({"top1": round(float(cm.top1), 4), "top5": round(float(cm.top5), 4),
           "macro_f1": round(float(macro_f1), 4), "n_test": len(y_true)},
          open(f"{CLS_DIR}/test_metrics.json", "w"), indent=1)
Path(f"{CLS_DIR}/classification_report.txt").write_text(report)
json.dump({"labels": [names[i] for i in labels],
           "matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist()},
          open(f"{CLS_DIR}/test_confusion.json", "w"))
print("-> saved test_metrics.json, classification_report.txt, test_confusion.json in", CLS_DIR)

## 6 - Weights + files to download
Everything for this run is in `/kaggle/working/runs/species_baseline/` (Output tab after a Commit).
The final `best.pt` is ~11 MB; a much larger one is an unfinished checkpoint.

In [ ]:
import os, shutil
for f in ["class_report.csv", "split_counts.json", "data_card.json"]:   # copy the data card next to the run
    if os.path.exists(f"{WORK}/cls_baseline/{f}"): shutil.copy(f"{WORK}/cls_baseline/{f}", CLS_DIR)
for f in ["weights/best.pt", "results.csv", "args.yaml", "test_metrics.json",
          "classification_report.txt", "test_confusion.json", "class_report.csv"]:
    p = f"{CLS_DIR}/{f}"
    print(f"OK      {p}  ({os.path.getsize(p) / 1e6:.1f} MB)" if os.path.exists(p) else f"MISSING {p}")